# Chapter 9 – Worked examples

This notebook runs the examples of [Chapter 9](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-2-circuits/ch09-ac-steady-state/) live. The code lives in the modules next to it
(`ac_transfer.py`, `rl_load.py`, `instantaneous_power.py`, `ac_nodal.py`); the cells only call them, so a
change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q matplotlib numpy pandas tzdata

In [ ]:
import ac_nodal
import ac_transfer
import instantaneous_power
import matplotlib.pyplot as plt
import numpy as np
import rl_load

## Power through a lossless reactance

Bus 1 at $V_1\angle\delta$, bus 2 at $V_2\angle 0$, reactance $X$. `transfer_power` evaluates
$P_{12} = V_1 V_2 \sin\delta / X$ and $Q_{12} = (V_1^2 - V_1 V_2 \cos\delta)/X$; `dc_estimate` is $\delta/X$.

In [ ]:
for delta in (5, 10, 30, 60, 90):
    p, q = ac_transfer.transfer_power(1.0, 1.0, 0.1, delta)
    p_dc = ac_transfer.dc_estimate(0.1, delta)
    print(f'delta = {delta:2d} deg  P = {p:6.3f} pu  Q = {q:6.3f} pu  DC estimate {p_dc:6.3f} pu')

The $P$–$\delta$ curve and its tangent at the origin. The DC model is the tangent: it has no maximum, while
the exact curve peaks at $V_1 V_2 / X$ for $\delta = 90^\circ$.

In [ ]:
delta = np.linspace(0, 180, 361)
p, _ = ac_transfer.transfer_power(1.0, 1.0, 0.1, delta)
plt.plot(delta, p, label='exact')
plt.plot(delta, ac_transfer.dc_estimate(0.1, delta), '--', label='DC: delta / X')
plt.ylim(0, 12)
plt.xlabel('delta (deg)')
plt.ylabel('P (pu)')
plt.legend()
plt.grid(True)
plt.show()

## The error of the DC approximation

`error_table` tabulates the exact power, the DC estimate and the relative error every 5 degrees. The check
values of the lab (0.5 % at $10^\circ$, 4.7 % at $30^\circ$, 21 % at $60^\circ$) are the `rel_error_pct` column.

In [ ]:
table = ac_transfer.error_table()
table[table.delta_deg.isin([10, 30, 60])]

## A load, its power and its power factor

The RL load of Exercise 1 as complex numbers: `impedance`, `load_power` and `power_factor` from `rl_load.py`.

In [ ]:
z = rl_load.impedance(10.0, 0.050, 50.0)
i, s = rl_load.load_power(230.0, z)
print(f'Z = {z:.2f} ohm   |I| = {abs(i):.2f} A   S = {s:.0f} VA   pf = {rl_load.power_factor(s):.3f}')

## Instantaneous power is not constant

`sample_power` samples $v(t)$, $i(t)$ and $p(t)$ over one period; the mean is $P$, the swing is $S$.

In [ ]:
t, v, i, p = instantaneous_power.sample_power(230.0, 10.0, 36.87)
mean, swing = instantaneous_power.mean_and_swing(p)
print(f'mean {mean:.1f} W, swing {swing:.1f} VA')
plt.plot(1000 * t, p)
plt.axhline(mean, color='k', linestyle='--')
plt.xlabel('t (ms)')
plt.ylabel('p(t) (W)')
plt.grid(True)
plt.show()

## Nodal analysis with complex admittances

The circuit of Exercise 5 through `ac_nodal.solve`: $Y V = I$ with a known voltage at node 1. The power the
two sources inject equals the power the two branches absorb.

In [ ]:
v, inj, i_branch = ac_nodal.solve(ac_nodal.FR, ac_nodal.TO, ac_nodal.Z_BRANCH, ac_nodal.V_SOURCE, ac_nodal.I_SOURCE)
print('node voltages  ', np.round(v, 2))
print('branch currents', np.round(i_branch, 2))
print('power balance  ', ac_nodal.power_balance(v, inj, i_branch, ac_nodal.Z_BRANCH))